# Attempt 002: Qwen3-4B joint preference classifier (train)

Read the question and both responses jointly, train a three-class head and LoRA adapters,
and compare with training-prior and majority baselines. The base model is the general
Qwen3-4B checkpoint, not a model fine-tuned on this competition. Gemma is not required.

All split assignments are fixed by explicit IDs from a separate prompt-grouped partition.
Training augments response swaps; evaluation averages both orientations. Model selection
uses development data. The confirmation partition is not evaluated in this notebook.

This training notebook uses internet for package setup and a model-download fallback.
The final competition inference notebook will have internet disabled.


In [ ]:
import os, sys, json, subprocess, base64, zlib, hashlib
from pathlib import Path
os.environ['TOKENIZERS_PARALLELISM']='false'
os.environ['HF_HUB_DISABLE_XET']='1'
os.environ['OMP_NUM_THREADS']='2'
os.environ['OPENBLAS_NUM_THREADS']='2'
os.environ['WANDB_DISABLED']='true'
WORK=Path('/kaggle/working')
MODULES=WORK/'bundled_code'
(MODULES/'arena').mkdir(parents=True,exist_ok=True)
(MODULES/'arena/__init__.py').write_text('')
os.environ['HF_HOME']=str(WORK/'hf_metadata_cache')
import torch
assert torch.cuda.is_available(), 'A GPU must be allocated for QLoRA'
print('CUDA devices:', [(torch.cuda.get_device_name(i), torch.cuda.get_device_properties(i).total_memory) for i in range(torch.cuda.device_count())])
import importlib.metadata
before={d.metadata['Name'].lower():d.version for d in importlib.metadata.distributions() if d.metadata['Name']}
subprocess.run([sys.executable,'-m','pip','install','-q','--disable-pip-version-check',
    'transformers==5.22.1','peft==0.21.2','accelerate==1.15.0','bitsandbytes==0.50.2'],check=True)
after={d.metadata['Name'].lower():d.version for d in importlib.metadata.distributions() if d.metadata['Name']}
changed={n:v for n,v in after.items() if before.get(n)!=v and not n.startswith(('torch','nvidia-'))}
changed.update({n:after[n] for n in ['transformers','peft','accelerate','bitsandbytes']})
runtime=[f'{n}=={v}' for n,v in sorted(changed.items())]
(WORK/'runtime_requirements.json').write_text(json.dumps(runtime,indent=2))
subprocess.run([sys.executable,'-m','pip','download','-q','--no-deps','--dest',str(WORK/'runtime_wheels'),*runtime],check=True)


In [ ]:
(MODULES/'arena/data.py').write_text('from __future__ import annotations\n\nimport hashlib\nimport json\nimport os\nimport re\nfrom pathlib import Path\n\nimport numpy as np\nimport pandas as pd\nfrom sklearn.model_selection import StratifiedGroupKFold\n\nROOT = Path(os.environ.get("ARENA_DATA_DIR", Path.home() / "database/llm_classification_finetuning"))\nCLASSES = ["winner_model_a", "winner_model_b", "winner_tie"]\nTEXT_COLUMNS = ["prompt", "response_a", "response_b"]\nSEED = 20261101\n\n\ndef parse_turns(value):\n    """Kaggle stores JSON arrays in CSV fields; preserve turn boundaries and nulls."""\n    if not isinstance(value, str):\n        return ""\n    try:\n        turns = json.loads(value)\n    except json.JSONDecodeError as exc:\n        raise ValueError("Expected a JSON-encoded conversation list") from exc\n    if not isinstance(turns, list):\n        raise ValueError("Expected a list of conversation turns")\n    return "\\n\\n[TURN]\\n\\n".join("" if turn is None else str(turn) for turn in turns)\n\n\ndef load_data():\n    train = pd.read_csv(ROOT / "raw/train.csv")\n    test = pd.read_csv(ROOT / "raw/test.csv")\n    for frame in (train, test):\n        for col in TEXT_COLUMNS:\n            frame[col] = frame[col].map(parse_turns)\n    assert train[CLASSES].isin([0, 1]).all().all()\n    assert train[CLASSES].sum(axis=1).eq(1).all()\n    assert train.id.is_unique and test.id.is_unique\n    return train, test\n\n\ndef prompt_groups(frame):\n    return frame.prompt.map(lambda s: hashlib.sha256(re.sub(r"\\s+", " ", s).strip().casefold().encode()).hexdigest())\n\n\ndef split_data(frame):\n    """8 train folds, 1 development fold, 1 untouched final holdout fold."""\n    y = frame[CLASSES].values.argmax(1)\n    groups = prompt_groups(frame)\n    fold = np.full(len(frame), -1)\n    cv = StratifiedGroupKFold(10, shuffle=True, random_state=SEED)\n    for k, (_, idx) in enumerate(cv.split(frame, y, groups)):\n        fold[idx] = k\n    parts = {"train": np.flatnonzero(fold >= 2), "dev": np.flatnonzero(fold == 1), "holdout": np.flatnonzero(fold == 0)}\n    for a, b in [("train", "dev"), ("train", "holdout"), ("dev", "holdout")]:\n        assert not set(groups.iloc[parts[a]]) & set(groups.iloc[parts[b]])\n    return y, parts, fold\n\n\ndef structural_features(frame):\n    """Only features available in test. Never use model identities or IDs."""\n    values = []\n    for col in TEXT_COLUMNS:\n        text = frame[col]\n        n = text.str.len().to_numpy()\n        words = text.str.count(r"\\S+").to_numpy()\n        values.append(np.column_stack([\n            np.log1p(n), np.log1p(words), np.log1p(text.str.count("\\n")),\n            np.log1p(text.str.count("```")), np.log1p(text.str.count(r"[!?]")),\n            np.log1p(text.str.count(r"(?m)^\\s*(?:[-*]|\\d+\\.)\\s")),\n            text.str.contains(r"(?i)\\b(?:sorry|cannot|can\'t|unable|as an ai)\\b", regex=True).astype(float),\n            text.str.count(r"[^\\x00-\\x7F]").to_numpy() / np.maximum(n, 1),\n            (words == 0).astype(float),\n        ]))\n    p, a, b = values\n    return np.column_stack([p, a, b, a-b, np.abs(a-b)]).astype(np.float32)\n\n\ndef swapped(frame):\n    out = frame.copy()\n    out[["response_a", "response_b"]] = frame[["response_b", "response_a"]].values\n    return out\n\n\ndef dense_pair_features(embeddings, stats):\n    p, a, b = embeddings\n    return np.column_stack([p, a, b, a-b, np.abs(a-b), a*b, p*a, p*b, stats]).astype(np.float32)\n\n\ndef swapped_pair_features(embeddings, stats):\n    p, a, b = embeddings\n    d = stats.shape[1] // 5\n    sp, sa, sb, delta, absolute = np.split(stats, [d, 2*d, 3*d, 4*d], axis=1)\n    return dense_pair_features((p, b, a), np.column_stack([sp, sb, sa, -delta, absolute]))\n\n\ndef json_write(path, value):\n    path = Path(path)\n    path.parent.mkdir(parents=True, exist_ok=True)\n    temp = path.with_suffix(path.suffix + ".tmp")\n    temp.write_text(json.dumps(value, indent=2, default=lambda x: x.item() if hasattr(x, "item") else str(x)) + "\\n")\n    temp.replace(path)\n')
print('Bundled arena/data.py')

In [ ]:
(MODULES/'arena/joint_data.py').write_text('"""Token-budgeted joint question/answer inputs and isolated attempt-002 splits."""\nfrom __future__ import annotations\nimport hashlib\nimport json\nimport random\nfrom pathlib import Path\n\nimport numpy as np\nimport pandas as pd\nimport torch\nfrom sklearn.model_selection import StratifiedGroupKFold\nfrom torch.utils.data import Dataset\n\nfrom .data import CLASSES, TEXT_COLUMNS, parse_turns, prompt_groups\n\nJOINT_SEED = 20261102\nSWAP_LABELS = np.array([1, 0, 2])\n\n\ndef phase_two_split(frame):\n    labels = frame[CLASSES].values.argmax(1)\n    groups = prompt_groups(frame)\n    folds = np.full(len(frame), -1, dtype=np.int8)\n    splitter = StratifiedGroupKFold(10, shuffle=True, random_state=JOINT_SEED)\n    for k, (_, idx) in enumerate(splitter.split(frame, labels, groups)):\n        folds[idx] = k\n    parts = {"train": np.flatnonzero(folds >= 2), "dev": np.flatnonzero(folds == 0),\n             "confirmation": np.flatnonzero(folds == 1)}\n    for a, b in [("train", "dev"), ("train", "confirmation"), ("dev", "confirmation")]:\n        assert not set(groups.iloc[parts[a]]) & set(groups.iloc[parts[b]])\n    return parts, folds\n\n\ndef split_from_ids(frame, manifest):\n    """Use explicit IDs to keep splits identical across sklearn/runtime versions."""\n    ids = frame.id.to_numpy()\n    parts = {name: np.flatnonzero(np.isin(ids, manifest[name])) for name in ["dev", "confirmation"]}\n    assert len(parts["dev"]) == len(manifest["dev"])\n    assert len(parts["confirmation"]) == len(manifest["confirmation"])\n    assert not set(manifest["dev"]) & set(manifest["confirmation"])\n    excluded = np.concatenate(list(parts.values()))\n    parts["train"] = np.flatnonzero(~np.isin(np.arange(len(frame)), excluded))\n    groups = prompt_groups(frame)\n    for a, b in [("train", "dev"), ("train", "confirmation"), ("dev", "confirmation")]:\n        assert not set(groups.iloc[parts[a]]) & set(groups.iloc[parts[b]])\n    return parts\n\n\ndef read_competition(raw):\n    raw = Path(raw)\n    train, test = pd.read_csv(raw / "train.csv"), pd.read_csv(raw / "test.csv")\n    for frame in (train, test):\n        for col in TEXT_COLUMNS:\n            frame[col] = frame[col].map(parse_turns)\n    return train, test\n\n\ndef head_tail(tokens, budget, marker):\n    if len(tokens) <= budget:\n        return tokens\n    if budget <= len(marker) + 2:\n        return tokens[:budget]\n    remaining = budget - len(marker)\n    first = (remaining + 1) // 2\n    return tokens[:first] + marker + tokens[-(remaining-first):]\n\n\nclass JointTokenizer:\n    """Preserve each field explicitly; truncate fields rather than dropping answer B."""\n    def __init__(self, tokenizer, max_length=1024):\n        self.tokenizer, self.max_length = tokenizer, max_length\n        encode = lambda s: tokenizer.encode(s, add_special_tokens=False)\n        self.prefix = encode("Compare both answers to the user request. Predict the human preference: answer A, answer B, or a tie.\\n\\n[USER REQUEST]\\n")\n        self.a_marker = encode("\\n\\n[ANSWER A]\\n")\n        self.b_marker = encode("\\n\\n[ANSWER B]\\n")\n        self.suffix = encode("\\n\\n[DECISION]\\n")\n        self.truncated = encode("\\n[TRUNCATED]\\n")\n        self.bos = [] if tokenizer.bos_token_id is None else [tokenizer.bos_token_id]\n        self.eos = [] if tokenizer.eos_token_id is None else [tokenizer.eos_token_id]\n        self.overhead = sum(map(len, [self.prefix, self.a_marker, self.b_marker, self.suffix, self.bos, self.eos]))\n        if max_length - self.overhead < 64:\n            raise ValueError("Context budget is too small for all three fields")\n\n    def combine(self, prompt, a, b):\n        available = self.max_length - self.overhead\n        prompt_budget = min(len(prompt), min(256, max(32, available // 4)))\n        answer_budget = available - prompt_budget\n        a_budget, b_budget = min(len(a), answer_budget // 2), min(len(b), answer_budget // 2)\n        unused = answer_budget - a_budget - b_budget\n        take = min(unused, len(a)-a_budget)\n        a_budget += take; unused -= take\n        b_budget += min(unused, len(b)-b_budget)\n        sequence = (self.bos + self.prefix + head_tail(prompt, prompt_budget, self.truncated)\n            + self.a_marker + head_tail(a, a_budget, self.truncated)\n            + self.b_marker + head_tail(b, b_budget, self.truncated) + self.suffix + self.eos)\n        assert len(sequence) <= self.max_length\n        return sequence\n\n    def encode_frame(self, frame):\n        original, swapped = [], []\n        totals = {name: 0 for name in TEXT_COLUMNS}\n        maxima = {name: 0 for name in TEXT_COLUMNS}\n        # Avoid retaining the entire untruncated corpus on each distributed rank.\n        for start in range(0,len(frame),256):\n            fields = []\n            for name in TEXT_COLUMNS:\n                tokens = self.tokenizer(frame[name].iloc[start:start+256].tolist(),\n                    add_special_tokens=False,truncation=False,padding=False)[\'input_ids\']\n                fields.append(tokens)\n                lengths = list(map(len,tokens))\n                totals[name] += sum(lengths)\n                maxima[name] = max(maxima[name],max(lengths,default=0))\n            for p,a,b in zip(*fields):\n                original.append(self.combine(p,a,b))\n                swapped.append(self.combine(p,b,a))\n        diagnostics = {name:{\'mean_tokens\':totals[name]/max(len(frame),1),\n                            \'max_tokens\':maxima[name]} for name in TEXT_COLUMNS}\n        return original, swapped, diagnostics\n\n\nclass PreferenceDataset(Dataset):\n    def __init__(self, original, swapped, labels=None, augment=False, force_swap=False):\n        self.original, self.swapped = original, swapped\n        self.labels, self.augment, self.force_swap = labels, augment, force_swap\n\n    def __len__(self):\n        return len(self.original)\n\n    def __getitem__(self, index):\n        flip = self.force_swap or (self.augment and random.random() < .5)\n        ids = (self.swapped if flip else self.original)[index]\n        item = {"input_ids": ids, "attention_mask": [1]*len(ids)}\n        if self.labels is not None:\n            y = int(self.labels[index])\n            item["labels"] = int(SWAP_LABELS[y]) if flip else y\n        return item\n\n\nclass PreferenceCollator:\n    def __init__(self, tokenizer):\n        self.tokenizer = tokenizer\n\n    def __call__(self, rows):\n        labels = [row["labels"] for row in rows] if "labels" in rows[0] else None\n        inputs = [{k:v for k,v in row.items() if k != "labels"} for row in rows]\n        batch = self.tokenizer.pad(inputs, padding=True, pad_to_multiple_of=8, return_tensors="pt")\n        if labels is not None:\n            batch["labels"] = torch.tensor(labels, dtype=torch.long)\n        return batch\n')
print('Bundled arena/joint_data.py')

In [ ]:
(MODULES/'arena/joint_train.py').write_text('"""QLoRA joint preference classification, with resumable checkpoints and swap evaluation."""\nfrom __future__ import annotations\nimport argparse\nimport hashlib\nimport json\nimport math\nimport os\nimport platform\nimport time\nfrom pathlib import Path\n\nimport numpy as np\nimport torch\nfrom scipy.optimize import minimize_scalar\nfrom sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score, log_loss\nfrom transformers import (AutoModelForSequenceClassification, AutoTokenizer,\n    BitsAndBytesConfig, EarlyStoppingCallback, Trainer, TrainerCallback, TrainingArguments, set_seed)\nfrom peft import LoraConfig, TaskType, get_peft_model, prepare_model_for_kbit_training\n\nfrom .data import CLASSES, json_write\nfrom .joint_data import (JointTokenizer, PreferenceDataset, PreferenceCollator,\n                         read_competition, split_from_ids)\n\n\ndef probabilities(logits):\n    logits = np.asarray(logits, dtype=np.float64)\n    logits -= logits.max(axis=1, keepdims=True)\n    exp = np.exp(logits)\n    return exp / exp.sum(axis=1, keepdims=True)\n\n\ndef metrics(labels, p):\n    assert p.shape == (len(labels), 3) and np.isfinite(p).all()\n    assert np.allclose(p.sum(1), 1, atol=1e-6)\n    return {"log_loss": float(log_loss(labels, p, labels=[0,1,2])),\n        "accuracy": float(accuracy_score(labels, p.argmax(1))),\n        "balanced_accuracy": float(balanced_accuracy_score(labels, p.argmax(1))),\n        "macro_f1": float(f1_score(labels, p.argmax(1), labels=[0,1,2], average="macro", zero_division=0)),\n        "brier": float(np.mean(np.sum((p-np.eye(3)[labels])**2,axis=1)))}\n\n\ndef temper(p, temperature):\n    return probabilities(np.log(np.clip(p, 1e-12, 1)) / temperature)\n\n\nclass PreferenceTrainer(Trainer):\n    """Mean example loss, with accumulation/DDP scaling owned by Trainer.\n\n    Qwen\'s generic classification forward accepts **kwargs but does not pass\n    num_items_in_batch to its loss. Trainer otherwise assumes that loss was\n    normalized over the accumulated global batch and scales it incorrectly.\n    """\n    def __init__(self, *args, group_training_by_length=False, **kwargs):\n        self.group_training_by_length = group_training_by_length\n        super().__init__(*args, **kwargs)\n        self.model_accepts_loss_kwargs = False\n        self.loss_is_scaled_for_ga = False\n\n    def _get_train_sampler(self, train_dataset=None):\n        if not self.group_training_by_length:\n            return super()._get_train_sampler(train_dataset)\n        from transformers.trainer_pt_utils import LengthGroupedSampler\n        dataset = self.train_dataset if train_dataset is None else train_dataset\n        lengths = [max(len(a),len(b)) for a,b in zip(dataset.original,dataset.swapped)]\n        # Group only training batches. Prediction ordering must stay sequential\n        # so original/swapped probabilities match the external row IDs and labels.\n        return LengthGroupedSampler(self.args.train_batch_size*self.args.gradient_accumulation_steps,\n                                    lengths=lengths)\n\n    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):\n        labels = inputs[\'labels\']\n        outputs = model(**{k: v for k, v in inputs.items() if k != \'labels\'})\n        loss = torch.nn.functional.cross_entropy(outputs.logits.float(), labels)\n        return (loss, outputs) if return_outputs else loss\n\n\nclass Journal(TrainerCallback):\n    def __init__(self, path, max_seconds):\n        self.path, self.max_seconds = Path(path), max_seconds\n        self.started = None\n        self.stopped_for_budget = False\n\n    def on_train_begin(self, args, state, control, **kwargs):\n        self.started = time.monotonic()\n\n    def on_log(self, args, state, control, logs=None, **kwargs):\n        if not state.is_world_process_zero:\n            return\n        record = {"step": state.global_step, "elapsed_seconds": time.monotonic()-self.started, **(logs or {})}\n        with self.path.open("a") as f:\n            f.write(json.dumps(record, default=str)+"\\n")\n        print("JOINT_PROGRESS", json.dumps(record, default=str), flush=True)\n        loss = (logs or {}).get("loss")\n        if loss is not None and not math.isfinite(loss):\n            raise RuntimeError("Non-finite training loss; stopping instead of consuming the run budget")\n\n    def on_step_end(self, args, state, control, **kwargs):\n        stop = bool(self.max_seconds and time.monotonic()-self.started > self.max_seconds)\n        if torch.distributed.is_available() and torch.distributed.is_initialized():\n            flag = torch.tensor(int(stop), device=torch.device("cuda",torch.cuda.current_device()))\n            torch.distributed.all_reduce(flag,op=torch.distributed.ReduceOp.MAX)\n            stop = bool(flag.item())\n        if stop:\n            self.stopped_for_budget = True\n            control.should_training_stop = True\n            control.should_save = True\n            control.should_evaluate = True\n        return control\n\n\ndef load_base_model(model_path, config, training):\n    cuda = torch.cuda.is_available()\n    rank = int(os.environ.get("LOCAL_RANK", "0"))\n    if cuda:\n        torch.cuda.set_device(rank)\n    native_bf16 = cuda and torch.cuda.get_device_capability(rank)[0] >= 8\n    dtype = torch.bfloat16 if native_bf16 else (torch.float16 if cuda else torch.float32)\n    quantize = config.get("quantize", True)\n    if quantize and not cuda:\n        raise RuntimeError("Full QLoRA requires a GPU; CPU is reserved for tiny correctness tests")\n    kwargs = dict(num_labels=3, id2label=dict(enumerate(CLASSES)), label2id={c:i for i,c in enumerate(CLASSES)},\n                  dtype=dtype, attn_implementation="sdpa", local_files_only=True)\n    if quantize:\n        kwargs["quantization_config"] = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",\n            bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype=dtype,\n            llm_int8_skip_modules=["score"])\n        kwargs["device_map"] = {"": rank}\n    model = AutoModelForSequenceClassification.from_pretrained(str(model_path), **kwargs)\n    model.config.use_cache = False\n    if quantize:\n        model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=training,\n            gradient_checkpointing_kwargs={"use_reentrant": False})\n    return model, dtype\n\n\ndef load_trainable_model(model_path, config):\n    model, dtype = load_base_model(model_path, config, training=True)\n    lora = LoraConfig(task_type=TaskType.SEQ_CLS, r=config["lora_rank"],\n        lora_alpha=config["lora_alpha"], lora_dropout=config["lora_dropout"], bias="none",\n        target_modules=config["target_modules"], modules_to_save=["score"])\n    model = get_peft_model(model, lora)\n    assert any("score" in n and p.requires_grad for n,p in model.named_parameters())\n    assert any("lora_" in n and p.requires_grad for n,p in model.named_parameters())\n    return model, dtype\n\n\ndef compute_metrics(prediction):\n    logits = prediction.predictions[0] if isinstance(prediction.predictions, tuple) else prediction.predictions\n    return metrics(prediction.label_ids, probabilities(logits))\n\n\ndef make_trainer(model, tokenizer, train_dataset, dev_dataset, config, out, pilot_steps=0):\n    cuda = torch.cuda.is_available()\n    bf16 = cuda and torch.cuda.get_device_capability()[0] >= 8\n    eval_steps = min(config["eval_steps"], max(1, pilot_steps//2)) if pilot_steps else config["eval_steps"]\n    world = int(os.environ.get("WORLD_SIZE", "1"))\n    total_steps = pilot_steps or math.ceil(config["epochs"]*len(train_dataset)/\n        (config["train_batch_size"]*config["gradient_accumulation_steps"]*world))\n    arguments = TrainingArguments(output_dir=str(out/"checkpoints"),\n        num_train_epochs=config["epochs"], max_steps=pilot_steps if pilot_steps else -1,\n        per_device_train_batch_size=config["train_batch_size"],\n        per_device_eval_batch_size=config["eval_batch_size"],\n        gradient_accumulation_steps=config["gradient_accumulation_steps"],\n        learning_rate=config["learning_rate"], weight_decay=config["weight_decay"],\n        warmup_steps=max(1,int(.05*total_steps)), lr_scheduler_type="cosine", max_grad_norm=1.,\n        gradient_checkpointing=cuda, gradient_checkpointing_kwargs={"use_reentrant":False},\n        bf16=bf16, fp16=cuda and not bf16, eval_strategy="steps", eval_steps=eval_steps,\n        save_strategy="steps", save_steps=eval_steps, save_total_limit=2,\n        load_best_model_at_end=True, metric_for_best_model="log_loss", greater_is_better=False,\n        logging_steps=5, logging_first_step=True, logging_nan_inf_filter=False,\n        report_to="none", seed=config["seed"], data_seed=config["seed"],\n        ddp_find_unused_parameters=False, ddp_timeout=1800, dataloader_num_workers=0, dataloader_pin_memory=cuda,\n        remove_unused_columns=False, optim="adamw_torch", disable_tqdm=True)\n    callbacks = [Journal(out/"training_progress.jsonl", config["max_train_seconds"])]\n    if not pilot_steps:\n        callbacks.append(EarlyStoppingCallback(early_stopping_patience=3))\n    return PreferenceTrainer(model=model, args=arguments, train_dataset=train_dataset, eval_dataset=dev_dataset,\n        group_training_by_length=config.get(\'group_training_by_length\',False),\n        processing_class=tokenizer, data_collator=PreferenceCollator(tokenizer),\n        compute_metrics=compute_metrics, callbacks=callbacks)\n\n\ndef predict_swapped(trainer, original, swapped, labels):\n    def one(force):\n        ds = PreferenceDataset(original, swapped, labels, force_swap=force)\n        logits = trainer.predict(ds).predictions\n        if isinstance(logits, tuple): logits = logits[0]\n        return probabilities(logits)\n    return (one(False)+one(True)[:,[1,0,2]])*.5\n\n\ndef run_training(raw, model_path, split_path, config, out, pilot_steps=0, resume=None):\n    out=Path(out);out.mkdir(parents=True,exist_ok=True)\n    if (out/"summary.json").exists():\n        raise RuntimeError("This training result already exists; use a separate job directory")\n    if config.get(\'preprocessing_sha256\'):\n        assert hashlib.sha256(Path(__file__).with_name(\'joint_data.py\').read_bytes()).hexdigest()==config[\'preprocessing_sha256\']\n    if config.get(\'split_ids_sha256\'):\n        assert hashlib.sha256(Path(split_path).read_bytes()).hexdigest()==config[\'split_ids_sha256\']\n    set_seed(config["seed"])\n    tokenizer=AutoTokenizer.from_pretrained(str(model_path),local_files_only=True,padding_side="right")\n    if tokenizer.pad_token_id is None:\n        tokenizer.pad_token=tokenizer.eos_token\n    frame,_=read_competition(raw)\n    parts=split_from_ids(frame,json.loads(Path(split_path).read_text()))\n    rng=np.random.default_rng(config["seed"])\n    train_idx=parts["train"]\n    if pilot_steps:\n        train_idx=np.sort(rng.choice(train_idx,min(2048,len(train_idx)),replace=False))\n    selection_idx=np.sort(rng.choice(parts["dev"],min(128 if pilot_steps else config["selection_dev_rows"],len(parts["dev"])),replace=False))\n    encoder=JointTokenizer(tokenizer,config["max_length"])\n    train_frame=frame.iloc[train_idx]\n    original,swapped,train_diagnostics=encoder.encode_frame(train_frame)\n    labels=train_frame[CLASSES].values.argmax(1)\n    training=PreferenceDataset(original,swapped,labels,augment=True)\n    dev_frame=frame.iloc[selection_idx]\n    dev_orig,dev_swap,dev_diagnostics=encoder.encode_frame(dev_frame)\n    dev_labels=dev_frame[CLASSES].values.argmax(1)\n    development=PreferenceDataset(dev_orig,dev_swap,dev_labels)\n    model,dtype=load_trainable_model(model_path,config)\n    model.config.pad_token_id=tokenizer.pad_token_id\n    model.config.problem_type="single_label_classification"\n    trainer=make_trainer(model,tokenizer,training,development,config,out,pilot_steps)\n    started=time.monotonic()\n    result=trainer.train(resume_from_checkpoint=resume)\n    train_seconds=time.monotonic()-started\n    trainer.save_model(str(out/"adapter"))\n    if trainer.is_world_process_zero():\n        tokenizer.save_pretrained(out/"adapter")\n        json_write(out/"training_config.json",config)\n    # The confirmation partition is never evaluated during model selection.\n    final_idx=selection_idx if pilot_steps else parts["dev"]\n    val_frame=frame.iloc[final_idx]\n    val_orig,val_swap,_=encoder.encode_frame(val_frame)\n    val_y=val_frame[CLASSES].values.argmax(1)\n    prediction_started=time.monotonic()\n    p=predict_swapped(trainer,val_orig,val_swap,val_y)\n    prediction_seconds=time.monotonic()-prediction_started\n    if trainer.is_world_process_zero():\n        temperature=minimize_scalar(lambda t: metrics(val_y,temper(p,t))["log_loss"],bounds=(.5,3.),method="bounded").x\n        np.savez_compressed(out/"dev_predictions.npz",ids=val_frame.id.to_numpy(),labels=val_y,probabilities=p)\n        world=int(os.environ.get("WORLD_SIZE","1"))\n        summary={"model_id":config["model_id"],"model_path":str(model_path),"pilot":bool(pilot_steps),\n            "config":config,"dtype":str(dtype),"world_size":world,"train_rows":len(train_idx),\n            "selection_dev_rows":len(selection_idx),"final_dev_rows":len(final_idx),\n            "confirmation_evaluated":False,"global_steps":trainer.state.global_step,\n            "epochs_completed":float(trainer.state.epoch),\n            "stopped_for_budget":any(getattr(c,\'stopped_for_budget\',False) for c in trainer.callback_handler.callbacks),\n            "adapter_sha256":hashlib.sha256((out/\'adapter/adapter_model.safetensors\').read_bytes()).hexdigest(),\n            "best_selection_dev_log_loss":trainer.state.best_metric,\n            "best_checkpoint":trainer.state.best_model_checkpoint,\n            "training_seconds":train_seconds,"seconds_per_optimizer_step":train_seconds/max(trainer.state.global_step,1),\n            "development_prediction_seconds":prediction_seconds,\n            "train_result":result.metrics,"development":metrics(val_y,p),\n            "temperature":float(temperature),"calibrated_development":metrics(val_y,temper(p,temperature)),\n            "trainable_parameters":model.get_nb_trainable_parameters()[0],\n            "total_parameters":model.get_nb_trainable_parameters()[1],\n            "peak_gpu_bytes":torch.cuda.max_memory_allocated() if torch.cuda.is_available() else None,\n            "diagnostics":{"train":train_diagnostics,"development":dev_diagnostics},\n            "software":{"python":platform.python_version(),"torch":torch.__version__}}\n        json_write(out/"summary.json",summary)\n        print("JOINT_RESULT",json.dumps(summary),flush=True)\n    trainer.accelerator.wait_for_everyone()\n\n\ndef main():\n    parser=argparse.ArgumentParser()\n    parser.add_argument("--raw",type=Path,required=True)\n    parser.add_argument("--model",type=Path,required=True)\n    parser.add_argument("--split",type=Path,required=True)\n    parser.add_argument("--config",type=Path,required=True)\n    parser.add_argument("--out",type=Path,required=True)\n    parser.add_argument("--pilot-steps",type=int,default=0)\n    parser.add_argument("--resume")\n    args=parser.parse_args()\n    run_training(args.raw,args.model,args.split,json.loads(args.config.read_text()),args.out,args.pilot_steps,args.resume)\n\n\nif __name__=="__main__":\n    main()\n')
print('Bundled arena/joint_train.py')

In [ ]:
CONFIG={'attempt': '002', 'model_id': 'Qwen/Qwen3-4B', 'model_source': 'qwen-lm/qwen-3/transformers/4b/1', 'max_length': 1024, 'seed': 20261102, 'lora_rank': 16, 'lora_alpha': 32, 'lora_dropout': 0.05, 'target_modules': ['q_proj', 'k_proj', 'v_proj', 'o_proj'], 'learning_rate': 5e-05, 'weight_decay': 0.01, 'epochs': 1, 'train_batch_size': 4, 'eval_batch_size': 4, 'gradient_accumulation_steps': 4, 'group_training_by_length': True, 'eval_steps': 200, 'selection_dev_rows': 1024, 'max_train_seconds': 21600, 'preprocessing_sha256': 'a2e047f0e732dd4b1da19fb18baf49ecda888e7602ba3fb353a39c327a93e672', 'split_ids_sha256': '066f7ab82a3f14464ce0ef67bfd8b8b34ed190679bfea5c1fea80abc3aa4ecd1'}
SPLIT_BYTES=zlib.decompress(base64.b64decode('...'))
assert hashlib.sha256(SPLIT_BYTES).hexdigest()=='066f7ab82a3f14464ce0ef67bfd8b8b34ed190679bfea5c1fea80abc3aa4ecd1'
(WORK/'split_ids.json').write_bytes(SPLIT_BYTES)
(WORK/'training_config.json').write_text(json.dumps(CONFIG,indent=2))
print(CONFIG)

In [ ]:
RAW=next(p for p in [Path('/kaggle/input/llm-classification-finetuning'),
    Path('/kaggle/input/competitions/llm-classification-finetuning')] if (p/'train.csv').exists())
MODEL_CANDIDATES=[Path('/kaggle/input/qwen-3/transformers/4b/1'),Path('/kaggle/input/models/qwen-lm/qwen-3/transformers/4b/1')]
MODEL=next((p for p in MODEL_CANDIDATES if (p/'config.json').exists()),None)
if MODEL is None:
    from huggingface_hub import HfApi, snapshot_download
    revision=HfApi().model_info(CONFIG['model_id']).sha
    MODEL=WORK/'base_model'
    snapshot_download(CONFIG['model_id'],revision=revision,local_dir=MODEL,
        allow_patterns=['*.json','*.safetensors','*.model','merges.txt','vocab.json'])
    CONFIG['hf_revision']=revision
    (WORK/'training_config.json').write_text(json.dumps(CONFIG,indent=2))
print('Pinned base model:', MODEL)
assert json.loads((MODEL/'config.json').read_text())['model_type']=='qwen3'


In [ ]:
environment=os.environ.copy()
environment['PYTHONPATH']=str(MODULES)
environment['PYTHONUNBUFFERED']='1'
processes=torch.cuda.device_count()
command=[sys.executable,'-m','torch.distributed.run','--standalone',f'--nproc_per_node={processes}',
    '--module','arena.joint_train','--raw',str(RAW),'--model',str(MODEL),
    '--split',str(WORK/'split_ids.json'),'--config',str(WORK/'training_config.json'),
    '--out',str(WORK/'joint_run')]
command += []
with (WORK/'training_console.log').open('w') as log:
    process=subprocess.Popen(command,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1,env=environment)
    for line in process.stdout:
        print(line,end='',flush=True);log.write(line);log.flush()
    return_code=process.wait()
if return_code:
    raise RuntimeError(f'Joint training exited with status {return_code}; see training_console.log')
summary=json.loads((WORK/'joint_run/summary.json').read_text())
summary
